# 07 — Triển khai Ứng dụng (AI Deployment)
**Dự án: HitRadar Pro | Phân hệ: EPIC 2 — MLOps và Triển khai Hệ thống**

---

## 1. MỤC TIÊU VÀ PHƯƠNG PHÁP LUẬN
Giai đoạn Triển khai (Deployment) chịu trách nhiệm tích hợp mô hình học máy vào môi trường ứng dụng thực tế. Khung kiến trúc ứng dụng được thiết kế theo mô hình client-server phân tách độc lập (decoupled microservices architecture):
1. **Dịch vụ Hậu cảnh (Backend API - FastAPI):** Đóng vai trò là thành phần tiếp nhận yêu cầu (request handler) và thực thi suy luận (inference engine). FastAPI sử dụng cơ chế bất đồng bộ (asynchronous) để tối ưu hóa thông lượng (throughput) và giảm thiểu độ trễ (latency).
2. **Dịch vụ Tiền cảnh (Frontend Dashboard - Streamlit):** Đóng vai trò là giao diện người dùng (User Interface). Cung cấp các công cụ tương tác trực quan (sliders, selectbox) để thiết lập không gian tham số đầu vào và gửi truy vấn dạng JSON đến Backend.

Quá trình khởi tạo mã nguồn trong notebook này sử dụng giao thức `%%writefile` nhằm tự động hóa việc xuất bản (publishing) các tập tin định dạng `.py`, đảm bảo tính thống nhất về mặt phiên bản giữa quá trình phát triển (development) và quá trình triển khai (production).


In [ ]:
import os
import shutil

# Thiết lập Thư mục Triển khai
APP_DIR = "hitradar_app"
os.makedirs(APP_DIR, exist_ok=True)
print(f"Trạng thái: Thư mục triển khai '{APP_DIR}/' đã được tạo lập.")

# Điều phối Khối tài nguyên (Artifact Routing)
try:
    shutil.copy('../3.6.modeling/xgboost_model.pkl', f'{APP_DIR}/xgboost_model.pkl')
    shutil.copy('../3.6.modeling/scaler.pkl', f'{APP_DIR}/scaler.pkl')
    print("Trạng thái: Di chuyển mô hình và bộ tiền xử lý thành công.")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy tập tin .pkl. Yêu cầu hoàn thành giai đoạn 06 (Modeling).")


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Quản trị Tài nguyên Triển khai (Artifact Management)

1. GIẢI THÍCH:
Mã nguồn thiết lập một Thư mục Môi trường Ứng dụng (App Directory) bằng module `os`. Sau đó, hệ thống sử dụng module `shutil` để di dời hai khối tài nguyên quý giá nhất từ giai đoạn Machine Learning: Mô hình Dự đoán (`xgboost_model.pkl`) và Bộ Định chuẩn (`scaler.pkl`) vào trung tâm của thư mục ứng dụng.

2. NHẬN XÉT:
Bước chuẩn bị này phản ánh sự am hiểu sâu sắc về chu trình MLOps (Machine Learning Operations). Việc đóng gói song song cả "Não bộ dự báo" (Mô hình) và "Hệ tiêu hóa dữ liệu" (Scaler) đảm bảo sự đồng bộ cấu trúc. Dữ liệu khi người dùng nhập vào ứng dụng thực tế sẽ đi qua một ống dẫn định chuẩn (Scaling Pipeline) y hệt như lúc mô hình được huấn luyện trong phòng thí nghiệm.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Bảo toàn tính nhất quán (Consistency Conservation) là chìa khóa của mọi ứng dụng AI. Nếu kỹ sư quên tích hợp `scaler.pkl` và để dữ liệu thô (ví dụ: `release_year = 2024`) đi thẳng vào một mô hình vốn chỉ hiểu các con số từ 0 đến 1, sự sai lệch phân phối (Distribution Shift) sẽ xảy ra lập tức, dẫn đến các dự báo rác (Garbage Predictions) phá hủy hoàn toàn uy tín của hệ thống phần mềm.

## 2. KIẾN TRÚC API VÀ GIAO THỨC SUY LUẬN (API & INFERENCE PROTOCOL)
Mã nguồn cấu trúc dịch vụ hậu cảnh được thiết lập tại tập tin `api.py`. Thành phần này tích hợp module xác thực dữ liệu tĩnh (static type validation) thông qua Pydantic.

In [ ]:
%%writefile hitradar_app/api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import pandas as pd
import numpy as np

app = FastAPI(title="HitRadar Inference API", version="1.0.0")

# --- Khởi tạo Khối Suy Luận ---
try:
    model = joblib.load("xgboost_model.pkl")
    scaler = joblib.load("scaler.pkl")
except Exception as e:
    raise RuntimeError(f"Lỗi tải mô hình tại API: {e}")

# --- Định nghĩa Lược đồ Dữ liệu Đầu vào (Schema Validation) ---
class TrackInput(BaseModel):
    duration_min: float = Field(..., ge=0.5, le=30.0)
    release_year: int = Field(..., ge=1900, le=2030)
    danceability: float = Field(..., ge=0.0, le=1.0)
    energy: float = Field(..., ge=0.0, le=1.0)
    loudness: float = Field(..., ge=-60.0, le=5.0)
    acousticness: float = Field(..., ge=0.0, le=1.0)
    instrumentalness: float = Field(..., ge=0.0, le=1.0)
    liveness: float = Field(..., ge=0.0, le=1.0)
    valence: float = Field(..., ge=0.0, le=1.0)
    tempo: float = Field(..., ge=20.0, le=250.0)
    time_signature: int = Field(..., ge=1, le=5)

@app.get("/")
def check_health():
    return {"status": "Operational", "service": "HitRadar API"}

@app.post("/predict")
def predict_popularity(track: TrackInput):
    try:
        # Tiền xử lý Dữ liệu Động (Online Preprocessing)
        data = pd.DataFrame([track.dict()])
        
        # Áp dụng hàm logarit tương đương bước xử lý ngoại tuyến (offline)
        data['speechiness_log'] = np.log1p(data['instrumentalness']) 
        
        FEATURES_ORDER = [
            'duration_min', 'release_year', 'danceability', 'energy', 'loudness', 
            'acousticness', 'liveness', 'valence', 'tempo', 'time_signature', 'speechiness_log'
        ]
        
        # Định chuẩn không gian (Scaling)
        data_scaled = scaler.transform(data[FEATURES_ORDER])
        
        # Gọi phương thức suy luận
        prediction = model.predict(data_scaled)[0]
        final_score = float(np.clip(prediction, 0.0, 100.0))
        
        # Phân loại hạng mục logic nghiệp vụ
        if final_score >= 70:
            tier = "High Potential (Tier 1)"
        elif final_score >= 50:
            tier = "Moderate Potential (Tier 2)"
        elif final_score >= 30:
            tier = "Low Potential (Tier 3)"
        else:
            tier = "Negligible Potential (Tier 4)"
            
        return {
            "predicted_score": final_score,
            "classification": tier,
            "status_code": 200
        }
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Kiến trúc API & Ràng buộc Giao thức (Validation Schema)

1. GIẢI THÍCH:
Khối mã xây dựng một Dịch vụ Hậu cảnh (Backend Service) tốc độ cao bằng kiến trúc ASGI của `FastAPI`. Nó thiết lập một điểm cuối (Endpoint) `POST /predict`. Đặc biệt, hệ thống ứng dụng `Pydantic BaseModel` để tạo ra một "Hàng rào Thép" (Validation Schema), quy định chặt chẽ miền giá trị hợp lệ (Valid Boundaries) cho từng thuộc tính đầu vào (ví dụ: `danceability` chỉ được phép nằm trong khoảng 0.0 - 1.0).

2. NHẬN XÉT:
Đây là thiết kế của một Kỹ sư Hệ thống Phần mềm (Software Systems Engineer) đích thực. Việc sử dụng FastAPI tích hợp Type Hints không chỉ sinh ra tài liệu API tự động (Swagger UI), mà còn tạo ra một lớp Tiền xử lý Trực tuyến (Online Preprocessing) cực kỳ thông minh. Bước gán biến đổi Logarit (`speechiness_log`) ngay tại hàm dự đoán đã tái hiện lại chính xác 100% logic xử lý ngoại tuyến (Offline Engineering) của Notebook 05.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Kiến trúc "Bảo vệ từ đầu vào" (Fail-fast Architecture) này là một tấm khiên chống lại Lỗ hổng Bảo mật (Vulnerability) và Dữ liệu bẩn. Bất kỳ yêu cầu (Request) nào cố tình đưa giá trị quá giới hạn (ví dụ: `tempo = -999`) sẽ bị FastAPI từ chối ngay ở lớp mạng (Network Layer) với lỗi 422 Unprocessable Entity, ngăn chặn hoàn toàn rủi ro sụp đổ hệ thống (System Crash) hoặc tràn bộ đệm (Buffer Overflow) của khối tính toán XGBoost.

## 3. THIẾT KẾ GIAO DIỆN TƯƠNG TÁC (FRONTEND DASHBOARD)
Xây dựng ứng dụng đơn trang (Single Page Application - SPA) dựa trên thư viện Streamlit nhằm mục đích hiển thị trực quan thông số dự đoán.

In [ ]:
%%writefile hitradar_app/app.py
import streamlit as st
import requests
import time

st.set_page_config(page_title="HitRadar Analytics Dashboard", layout="wide")

st.markdown("## Công cụ Phân tích Độ Phổ biến Âm thanh (Audio Popularity Analyzer)")
st.markdown("Thiết lập các tham số để cấu hình vec-tơ đặc trưng đầu vào.")

st.markdown("---")
col_1, col_2, col_3 = st.columns(3)

with col_1:
    st.markdown("#### Metadata & Thời gian")
    release_year = st.slider("Năm phát hành (Release Year)", 1980, 2025, 2024)
    duration_min = st.slider("Thời lượng (Duration in minutes)", 1.0, 10.0, 3.2, step=0.1)
    tempo = st.slider("Tốc độ Nhịp (Tempo - BPM)", 50, 200, 122)
    time_signature = st.selectbox("Nhịp phách (Time Signature)", [3, 4, 5], index=1)

with col_2:
    st.markdown("#### Động lực học (Dynamics)")
    danceability = st.slider("Danceability", 0.0, 1.0, 0.75, step=0.01)
    energy = st.slider("Energy", 0.0, 1.0, 0.85, step=0.01)
    valence = st.slider("Valence", 0.0, 1.0, 0.65, step=0.01)
    liveness = st.slider("Liveness", 0.0, 1.0, 0.10, step=0.01)

with col_3:
    st.markdown("#### Cấu trúc Âm thanh (Acoustics)")
    acousticness = st.slider("Acousticness", 0.0, 1.0, 0.15, step=0.01)
    instrumentalness = st.slider("Instrumentalness", 0.0, 1.0, 0.00, step=0.01)
    loudness = st.slider("Loudness (dB)", -25.0, 0.0, -5.5, step=0.5)

st.markdown("---")

if st.button("Tiến hành Phân tích (Run Inference)", use_container_width=True):
    payload = {
        "duration_min": duration_min, "release_year": release_year,
        "danceability": danceability, "energy": energy, "loudness": loudness,
        "acousticness": acousticness, "instrumentalness": instrumentalness,
        "liveness": liveness, "valence": valence, "tempo": tempo,
        "time_signature": time_signature
    }
    
    with st.spinner('Đang tính toán suy luận (Computing Inference)...'):
        time.sleep(0.5) 
        
        try:
            response = requests.post("http://localhost:8000/predict", json=payload)
            if response.status_code == 200:
                result = response.json()
                score = result['predicted_score']
                tier = result['classification']
                
                st.success(f"### Kết quả Suy luận: {score:.4f} / 100")
                st.info(f"**Phân loại Cấp độ (Classification Tier):** {tier}")
            else:
                st.error("Lỗi từ Dịch vụ Hậu cảnh.")
        except requests.exceptions.ConnectionError:
            st.error("Lỗi Kết nối (Connection Error): Không thể thiết lập TCP connection với FastAPI endpoint. Đảm bảo cổng 8000 đang được lắng nghe.")


### Giải thích, Nhận xét & Đánh giá Chuyên sâu: Thiết kế Giao diện Khách hàng (Frontend UI)

1. GIẢI THÍCH:
Ứng dụng thư viện `Streamlit` để triển khai Ứng dụng Đơn trang (Single Page Application). Giao diện được cấu trúc theo hệ thống Lưới 3 cột (3-Column Grid System) tối giản, phân chia các cụm tham số đầu vào (Inputs) thành 3 nhóm logic: Metadata, Động lực học (Dynamics), và Cấu trúc Âm thanh (Acoustics). Module `requests` được khởi tạo để làm bộ truyền tin HTTP (HTTP Client), đẩy gói dữ liệu JSON tới FastAPI.

2. NHẬN XÉT:
Giao diện người dùng (UI) bám sát triết lý Thiết kế Tương tác Trực quan (Interactive Design). Việc chuyển đổi các biến số khô khan thành Thanh trượt (Sliders) với các tham số giới hạn an toàn (Min/Max values) giúp người dùng không có kiến thức kỹ thuật (như Nhạc sĩ, Nhà sản xuất) dễ dàng thao tác mô phỏng (Simulation). Hiệu ứng chờ (Spinner) và phản hồi Trạng thái (Success/Error blocks) mang lại một Trải nghiệm Người dùng (UX) hiện đại và chuyên nghiệp.

3. ĐÁNH GIÁ (HIGH IMPACT):
Sự tách bạch hoàn toàn giữa Giao diện Khách (Frontend Streamlit) và Lõi Máy chủ (Backend FastAPI) hoàn thiện kiến trúc Microservices kinh điển. Nó đem lại Khả năng Mở rộng (Scalability) vô hạn: Nhóm Kỹ sư Giao diện có thể thoải mái sửa đổi, thay màu sắc, làm lại giao diện bằng ReactJS hay VueJS trên Mobile App mà không cần phải can thiệp hay sửa đổi bất kỳ dòng code nào của Hệ thống Trí tuệ Nhân tạo đang chạy ngầm phía sau.

## 4. HƯỚNG DẪN KHỞI TẠO HỆ THỐNG
Yêu cầu cấp phát 2 phiên (sessions) terminal độc lập trong môi trường máy chủ cục bộ (local host).

**Bước 1: Khởi động Dịch vụ API (Port 8000)**
```bash
cd hitradar_app
uvicorn api:app --reload
```

**Bước 2: Khởi động Ứng dụng Bảng điều khiển**
```bash
cd hitradar_app
streamlit run app.py
```

---
# KẾT LUẬN GIAI ĐOẠN EPIC 2
Quy trình kỹ thuật (Engineering Pipeline) đã được thực thi trọn vẹn thông qua các module chuẩn hóa: Từ khâu khai phá dữ liệu số lượng lớn (EDA), tinh chỉnh đặc trưng vĩ mô (Feature Engineering), huấn luyện các mô hình thống kê dự báo (Machine Learning), cho đến khâu đóng gói triển khai phần mềm (MLOps Deployment). Kiến trúc này đáp ứng đầy đủ các tiêu chí thực nghiệm, có thể dễ dàng bảo trì và tích hợp hoặc chuyển đổi nền tảng (platform agnosticism) khi có yêu cầu.
